# Azula — merge LoRA for Ollama (no training)

GPU gerekmez. Runtime → Run all.

Yerel dosya: `adapters/azula-incident/azula-lora-only.zip` (~29 MB). İlk hücrede yükle.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # azula-lora-only.zip seç
names = list(uploaded.keys())
print("yüklenen:", names)
zip_path = Path("/content") / names[0]
print("kullanılacak:", zip_path, "bytes=", zip_path.stat().st_size)

In [ ]:
!pip install -q peft transformers accelerate
import zipfile, os
from pathlib import Path

# İlk hücrede yüklenen dosya. Glob kullanma — /content'te başka bozuk zip olabilir.
zip_path = Path("/content") / list(uploaded.keys())[0]
print("açılan:", zip_path, zip_path.stat().st_size)
assert zip_path.stat().st_size > 1_000_000, "zip çok küçük / eksik yüklendi, tekrar yükle"

adapter = Path("/content/adapter")
adapter.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(adapter)
print("adapter files:", os.listdir(adapter))
assert "adapter_model.safetensors" in os.listdir(adapter)

In [ ]:
import torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_id = "Qwen/Qwen2.5-1.5B-Instruct"
adapter = "/content/adapter"
tok = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    base_id, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True
)
merged = PeftModel.from_pretrained(base, adapter).merge_and_unload()
if hasattr(merged.config, "quantization_config"):
    merged.config.quantization_config = None
out = Path("/content/merged-fp16")
out.mkdir(exist_ok=True)
merged.save_pretrained(out, safe_serialization=True)
tok.save_pretrained(out)
print("merged", out)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!zip -r /content/drive/MyDrive/azula-merged-fp16.zip /content/merged-fp16
print("Drive: MyDrive/azula-merged-fp16.zip")